# ETL Transform: News

This notebook runs the **news ETL pipeline**: ingest from Postgres → transform → save to `financial_news_transformed` → publish to S3.

**Two modes (set `USE_AGENTIC_ONLY` in the next cell):**
- **VADER (default)**: transform with sentiment (VADER), intent, keywords, tickers; then save and S3.
- **Agentic only**: skip VADER; run LLM-based financial metrics extraction only; then save and S3.

**S3 upload modes:**
- **Per-article**: one CSV per article at `news/crypto/[agentic=true|false/]year=.../.../format=csv/{id}.csv`
- **Batch (run)**: one CSV per run at `news/transformed/crypto/.../batch=run/...` (by run time)
- **Batch (year/month/week/day)**: partitioned by **article issue date** under `news/transformed/crypto/year=.../month=.../day=...` etc.

Set `AWS_NEWS_BUCKET` or `AWS_DEFAULT_BUCKET` in `.env` for S3 uploads. For agentic mode, set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` and optionally `LLM_PROVIDER` (default: openai).

In [1]:
import sys
from pathlib import Path

# Resolve project root: run from repo root or notebooks/etl/
_cwd = Path(".").resolve()
project_root = _cwd if (_cwd / "src").is_dir() else (_cwd.parent.parent if _cwd.name == "etl" else _cwd)
src_path = project_root / "src"
if src_path.is_dir():
    sys.path.insert(0, str(project_root))
    sys.path.insert(0, str(src_path))
else:
    raise FileNotFoundError(f"Expected src at {src_path}. Run from repo root or notebooks/etl/.")

import pandas as pd

In [2]:
# Options: set to True to skip VADER and use only agentic (LLM) enrichment
USE_AGENTIC_ONLY = True
SINCE = "2026-08-30"
UNTIL = "2026-09-01"
# For agentic-only: limit rows (None = no limit; set e.g. 10 for a quick test)
AGENTIC_MAX_ROWS = None
# Save every N rows (None = save all at end; 31 or 50 = save after each chunk for resilience)
AGENTIC_SAVE_EVERY_N_ROWS = None
# S3: per-article and/or batch (same for both modes)
UPLOAD_S3_PER_ARTICLE = True
UPLOAD_S3_BATCH = ["day"] #["run", "week", "month", "year", "day"]

In [3]:
# Run news ETL: either agentic-only (skip VADER) or VADER transform; then save to Postgres and S3.
# Agentic: saves all transformed rows (including any with llm_error). Query llm_error column later to retry/fix.

from datetime import datetime
from pipelines.etl_transform import (
    run_news_etl,
    save_transformed_news_to_postgres,
    _ensure_news_transformed_table,
    agentic_result_has_failures,
    build_s3_key_news_per_article,
    upload_news_batches_to_s3,
    upload_dataframe_to_s3_key,
)
from config.settings import get_settings
from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries as q

if USE_AGENTIC_ONLY:
    from pipelines.etl_cli import ingest_news
    from agents.registry import get_llm_client
    from agents.transforms.agentic_transform import AgenticTextEnricher, FinancialMetricsTask
    from storage.cloud.CloudStorage import CloudStorageProvider

    df = ingest_news(since=SINCE, until=UNTIL)
    if df is None or df.empty:
        transformed_df = pd.DataFrame()
        print("No news data to process")
    else:
        # Show daterange of rows being transformed
        if "datetime" in df.columns:
            dt = pd.to_datetime(df["datetime"], errors="coerce")
            valid = dt.notna()
            if valid.any():
                mn, mx = dt.loc[valid].min(), dt.loc[valid].max()
                daterange = f"{mn.date()} to {mx.date()}" if hasattr(mn, "date") else f"{str(mn)[:10]} to {str(mx)[:10]}"
                print(f"Transforming {len(df)} rows (daterange: {daterange})")
        settings = get_settings()
        provider = getattr(settings.agent, "provider", "openai")
        transformed_df = pd.DataFrame()
        try:
            client = get_llm_client(provider)
            enricher = AgenticTextEnricher(client=client, task=FinancialMetricsTask())
            to_process = min(len(df), AGENTIC_MAX_ROWS) if AGENTIC_MAX_ROWS is not None else len(df)
            chunk_size = AGENTIC_SAVE_EVERY_N_ROWS

            if chunk_size is None or chunk_size <= 0:
                # Enrich all, then save and upload once at the end
                transformed_df = enricher.enrich_dataframe(df, max_rows=AGENTIC_MAX_ROWS)
                if not transformed_df.empty:
                    conn = PgConn(q.FINANCIAL_NEWS_TABLE_NAME)
                    _ensure_news_transformed_table(conn)
                    n = save_transformed_news_to_postgres(conn, transformed_df, agentic_enabled=True)
                    conn.close_connection()
                    print(f"Saved {n} rows to {q.FINANCIAL_NEWS_TRANSFORMED_TABLE_NAME}")
                    bucket = getattr(settings.aws, "news_bucket", None) or settings.aws.default_bucket
                    if (UPLOAD_S3_PER_ARTICLE or UPLOAD_S3_BATCH) and not bucket:
                        print(
                            "S3 upload skipped: set AWS_NEWS_BUCKET or AWS_DEFAULT_BUCKET in .env, "
                            "then restart Jupyter."
                        )
                    if bucket and (UPLOAD_S3_PER_ARTICLE or UPLOAD_S3_BATCH):
                        aws = CloudStorageProvider.AWS()
                        if UPLOAD_S3_PER_ARTICLE:
                            for _, row in transformed_df.iterrows():
                                aid = str(row.get("id", ""))
                                dt_str = row.get("datetime")
                                try:
                                    dt = pd.to_datetime(dt_str) if dt_str else datetime.utcnow()
                                except Exception:
                                    dt = datetime.utcnow()
                                key = build_s3_key_news_per_article(aid, dt, agentic=True)
                                upload_dataframe_to_s3_key(aws.s3_client, bucket, key, pd.DataFrame([row]))
                            print(f"Uploaded {len(transformed_df)} per-article CSVs to s3://{bucket}/")
                        if UPLOAD_S3_BATCH:
                            upload_news_batches_to_s3(aws.s3_client, bucket, transformed_df, UPLOAD_S3_BATCH, agentic=True)
                            print(f"Uploaded batches {UPLOAD_S3_BATCH} to s3://{bucket}/")
            else:
                # Process and save in chunks (resilient: partial progress persisted if run stops)
                conn = PgConn(q.FINANCIAL_NEWS_TABLE_NAME)
                _ensure_news_transformed_table(conn)
                bucket = getattr(settings.aws, "news_bucket", None) or settings.aws.default_bucket
                if (UPLOAD_S3_PER_ARTICLE or UPLOAD_S3_BATCH) and not bucket:
                    print(
                        "S3 upload skipped: set AWS_NEWS_BUCKET or AWS_DEFAULT_BUCKET in .env, "
                        "then restart Jupyter."
                    )
                aws = CloudStorageProvider.AWS() if bucket and (UPLOAD_S3_PER_ARTICLE or UPLOAD_S3_BATCH) else None
                total_saved = 0
                chunks_out = []
                for start in range(0, to_process, chunk_size):
                    end = min(start + chunk_size, to_process)
                    chunk_df = df.iloc[start:end].copy()
                    enriched = enricher.enrich_dataframe(chunk_df, max_rows=None)
                    if enriched.empty:
                        continue
                    n = save_transformed_news_to_postgres(conn, enriched, agentic_enabled=True)
                    total_saved += n
                    chunks_out.append(enriched)
                    if bucket and UPLOAD_S3_PER_ARTICLE:
                        for _, row in enriched.iterrows():
                            aid = str(row.get("id", ""))
                            dt_str = row.get("datetime")
                            try:
                                dt = pd.to_datetime(dt_str) if dt_str else datetime.utcnow()
                            except Exception:
                                dt = datetime.utcnow()
                            key = build_s3_key_news_per_article(aid, dt, agentic=True)
                            upload_dataframe_to_s3_key(aws.s3_client, bucket, key, pd.DataFrame([row]))
                    print(f"  Chunk {start}-{end}: saved {n} rows (total so far: {total_saved})")
                conn.close_connection()
                transformed_df = pd.concat(chunks_out, ignore_index=True) if chunks_out else pd.DataFrame()
                print(f"Saved {total_saved} rows to {q.FINANCIAL_NEWS_TRANSFORMED_TABLE_NAME}")
                if bucket:
                    if UPLOAD_S3_PER_ARTICLE:
                        print(f"Uploaded {total_saved} per-article CSVs to s3://{bucket}/")
                    if UPLOAD_S3_BATCH and not transformed_df.empty:
                        upload_news_batches_to_s3(aws.s3_client, bucket, transformed_df, UPLOAD_S3_BATCH, agentic=True)
                        print(f"Uploaded batches {UPLOAD_S3_BATCH} to s3://{bucket}/")

            # Report any per-row llm_errors (all rows were saved; query llm_error in DB to retry later)
            if not transformed_df.empty and agentic_result_has_failures(transformed_df):
                err_series = transformed_df["llm_error"].fillna("").astype(str).str.strip()
                failed = err_series != ""
                n_failed = int(failed.sum())
                print(f"Note: {n_failed} / {len(transformed_df)} rows have llm_error set (saved anyway; query llm_error to retry).")
                for msg, count in transformed_df.loc[failed, "llm_error"].value_counts().items():
                    print(f"  [{count}] {msg}")
        except (KeyError, ValueError) as e:
            print(f"Agentic skipped (LLM not configured): {e}")
        except Exception as e:
            print(f"Agentic failed: {e}")
            transformed_df = pd.DataFrame()

    print(f"Agentic: transformed {len(transformed_df)} articles")
else:
    transformed_df = run_news_etl(
        since=SINCE,
        until=UNTIL,
        news_bucket=None,
        save_to_postgres=True,
        upload_s3_per_article=UPLOAD_S3_PER_ARTICLE,
        upload_s3_batch=UPLOAD_S3_BATCH if UPLOAD_S3_BATCH else None,
        sentiment_backend="vader",
        extract_tickers=True,
    )
    print(f"VADER: transformed {len(transformed_df)} articles")

Connection to the database successful!
Table name set to: financial_news_241118
Connection closed.
Transforming 93 rows (daterange: 2026-08-31 to 2026-09-01)
Agentic enrichment: processing 93 rows (daterange: 2026-08-31 to 2026-09-01, progress every 4 rows)
  progress: 4/93 rows
  progress: 8/93 rows
  progress: 12/93 rows
  progress: 16/93 rows
  progress: 20/93 rows
  progress: 24/93 rows
  progress: 28/93 rows
  progress: 32/93 rows
  progress: 36/93 rows
  progress: 40/93 rows
  progress: 44/93 rows
  progress: 48/93 rows
  progress: 52/93 rows
  progress: 56/93 rows
  progress: 60/93 rows
  progress: 64/93 rows
  progress: 68/93 rows
  progress: 72/93 rows
  progress: 76/93 rows
  progress: 80/93 rows
  progress: 84/93 rows
  progress: 88/93 rows
  progress: 92/93 rows
  progress: 93/93 rows
Agentic enrichment done: 93 rows
Connection to the database successful!
Table name set to: financial_news_241118
Table name set to: financial_news_transformed
Executing query: 
            CRE

In [4]:
# Inspect agentic llm_error (run this when you see "Agentic enrichment had errors...")
if not transformed_df.empty and "llm_error" in transformed_df.columns:
    err_series = transformed_df["llm_error"].fillna("").astype(str).str.strip()
    failed = err_series != ""
    n_failed = failed.sum()
    if n_failed:
        print(f"Rows with llm_error: {n_failed} / {len(transformed_df)}")
        print("\n--- Unique llm_error messages (count) ---")
        print(transformed_df.loc[failed, "llm_error"].value_counts().to_string())
        print("\n--- Sample rows with errors (id, headline, llm_error) ---")
        sample = transformed_df.loc[failed, ["id", "headline", "llm_error"]].head(10)
        for _, row in sample.iterrows():
            print(f"  id: {row['id']}")
            print(f"  headline: {(str(row['headline'])[:60])}...")
            print(f"  llm_error: {row['llm_error']}")
            print()
    else:
        print("No rows have llm_error set.")
else:
    print("No transformed_df or no llm_error column to inspect.")

No transformed_df or no llm_error column to inspect.


In [5]:
# Inspect transformed output
if not transformed_df.empty:
    display(transformed_df.head())
    print(transformed_df.columns.tolist())

,id,source,headline,href,summary,content,author,minsread,datetime,created_at,...,llm_immediacy,llm_impact_horizon,llm_confidence,llm_novelty_score,llm_sentiment_label,llm_impact_level,llm_signal,llm_actionable,llm_sectors,llm_key_facts
0,2135953138009737832,TheStreet,92-year-old burger chain's Bitcoin strategy de...,https://finance.yahoo.com/markets/crypto/artic...,,"Steak 'n Shake, a burger chain founded in 1934...",Bibhu Pattnaik,2 min read,2026-08-31 19:20:00,2026-09-01 03:18:25.737960,...,0.8,short_term,0.9,0.7,positive,high,bullish,True,"[equities, crypto]",[Steak 'n Shake reported a 13.8% same-store sa...
1,1027768243097796770,BeInCrypto,"A 71,000% Profit Surge Is Taking Longsys to Ho...",https://finance.yahoo.com/markets/stocks/artic...,,Hong Kong skyline representing Moonshot AI's p...,Kamina Bashir,2 min read,2026-08-31 12:39:57,2026-09-01 03:18:25.899054,...,0.9,short_term,0.85,0.7,positive,high,bullish,True,"[equities, other]",[Longsys seeking up to HK$6.28 billion from Ho...
2,4280933580458620790,CryptoProwl,Arbitrator Rules Gemini Not At Fault For Colla...,https://finance.yahoo.com/markets/crypto/artic...,,Arbitrator Rules Gemini Not At Fault For Colla...,Editorial Staff,2 min read,2026-08-31 14:22:00,2026-09-01 03:18:25.865915,...,0.6,short_term,0.7,0.1,positive,medium,bullish,True,[crypto],[Arbitrator ruled Gemini not at fault for Earn...
3,3634046561881663597,BeInCrypto,BeInCrypto Partners with TOKEN2049 Singapore 2026,https://finance.yahoo.com/markets/crypto/artic...,,Photo by BeInCrypto\nBeInCrypto is attending T...,Iva Belamaric,2 min read,2026-08-31 15:52:52,2026-09-01 03:18:25.811585,...,0.5,short_term,0.7,0.1,positive,medium,bullish,True,"[crypto, equities]",[BeInCrypto is an official media partner for T...
4,3516162197145226587,InvestorsHub,Bernstein’s Chhugani forecasts Bitcoin could r...,https://finance.yahoo.com/markets/crypto/artic...,,Bernstein expects Bitcoin (COIN:BTCUSD) to rea...,Fiona Craig,2 min read,2026-08-31 15:22:16,2026-09-01 03:18:25.829605,...,0.6,long_term,0.85,0.5,positive,high,bullish,True,[crypto],"[Bitcoin could reach $300,000 by 2029, Forecas..."


['id', 'source', 'headline', 'href', 'summary', 'content', 'author', 'minsread', 'datetime', 'created_at', 'llm_financial_metrics', 'llm_entities', 'llm_ticker', 'llm_event_type', 'llm_overall_sentiment', 'llm_forward_sentiment', 'llm_surprise_score', 'llm_risk_score', 'llm_uncertainty_score', 'llm_impact_strength', 'llm_immediacy', 'llm_impact_horizon', 'llm_confidence', 'llm_novelty_score', 'llm_sentiment_label', 'llm_impact_level', 'llm_signal', 'llm_actionable', 'llm_sectors', 'llm_key_facts']


## Optional: Per-article only (no batch)
Use when you want only one CSV per article in S3.

In [6]:
# transformed_per_article = run_news_etl(
#     since="2026-06-26",
#     until="2026-06-27",
#     save_to_postgres=True,
#     upload_s3_per_article=True,
#     upload_s3_batch=None,
# )

## Optional: Batch only (no per-article)
Use when you want a single CSV per run, or per week/month/year.

In [7]:
# transformed_batch = run_news_etl(
#     date="2026-01-27",
#     save_to_postgres=True,
#     upload_s3_per_article=False,
#     upload_s3_batch=["run", "month"],
# )